# Compliance RAG — exploration / demo

This notebook is for interactively exploring the pipeline, not the app's entry point.

The actual logic lives in:
- `ingest.py` — load PDFs, chunk, embed, persist to Chroma
- `retrieve.py` — hybrid (BM25 + semantic) retrieval and cross-encoder reranking
- `generate.py` — prompt construction and the Groq LLM call
- `main.py` — thin CLI that wires the above together (`python main.py "your question"`)

Run `python ingest.py` once from the terminal to build the index instead of using the cells below, unless you specifically want to step through ingestion here.

In [1]:
%pip install langchain-huggingface langchain-chroma

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from ingest import load_and_chunk, save_chunks, build_vectorstore
from retrieve import load_vectorstore, load_all_chunks, build_hybrid_retriever, search
from generate import generate_answer

In [ ]:
# One-time ingestion: load PDFs, chunk them, and persist the chunk corpus.
# Skip this cell if difc_chunks.pkl already exists.
chunks = load_and_chunk()
save_chunks(chunks)

In [ ]:
# One-time embedding: build and persist the Chroma vector store.
# Skip this cell on future runs — it's already persisted to disk in difc_chroma_db/.
vectorstore = build_vectorstore(chunks)

In [ ]:
# Demo: ask a question against the already-built index.
vectorstore = load_vectorstore()
all_chunks = load_all_chunks()
retriever = build_hybrid_retriever(vectorstore, all_chunks)

query = "What are the KYC requirements for a licensed firm under DIFC regulations?"
top_chunks = search(retriever, query)

answer = generate_answer(query, top_chunks)
print(answer)